# Register the Customer H2O Model

Register only the validated customer bundle as an immutable Azure ML custom model and verify its version and provenance tags.

**Source:** Adapted from this repository's H2O onboarding notebook and the Azure ML model asset examples.

In [ ]:
from pathlib import Path
import os
import sys

from azure.ai.ml import MLClient
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import Model
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / ".env.example").is_file() and (candidate / "pipelines").is_dir():
        WORKSHOP_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the workshop folder")
load_dotenv(WORKSHOP_ROOT / ".env", override=True)

model_value = Path(os.environ["H2O_CUSTOMER_MODEL_PATH"])
MODEL_PATH = model_value if model_value.is_absolute() else WORKSHOP_ROOT / model_value
BUNDLE_DIR = MODEL_PATH.resolve().parent
MODEL_NAME = os.environ["H2O_MODEL_NAME"]
MODEL_VERSION = os.environ["H2O_MODEL_VERSION"]
REGISTER = os.getenv("REGISTER_H2O_MODEL", "false").lower() in {"1", "true", "yes"}

sys.path.insert(0, str(WORKSHOP_ROOT / "src/h2o"))
from validate_bundle import validate_bundle
summary = validate_bundle(BUNDLE_DIR, os.environ["H2O_VERSION"])

credential = AzureCliCredential(tenant_id=os.getenv("AZURE_TENANT_ID") or None)
ml_client = MLClient(credential, os.environ["AZURE_SUBSCRIPTION_ID"], os.environ["AZURE_RESOURCE_GROUP"], os.environ["AZUREML_WORKSPACE_NAME"])
model_definition = Model(
    name=MODEL_NAME,
    version=MODEL_VERSION,
    type=AssetTypes.CUSTOM_MODEL,
    path=str(BUNDLE_DIR),
    description="Validated customer H2O binary-model bundle",
    tags={
        "workshop": "azureml-h2o",
        "model_format": summary["model_format"],
        "h2o_version": summary["h2o_version"],
        "golden_validation": "passed",
    },
)

if REGISTER:
    registered_model = ml_client.models.create_or_update(model_definition)
    verified_model = ml_client.models.get(MODEL_NAME, MODEL_VERSION)
    assert verified_model.tags["golden_validation"] == "passed"
    print(f"Registered model: {verified_model.name}:{verified_model.version}")
else:
    print(f"Validated model definition: {MODEL_NAME}:{MODEL_VERSION}")
    print("Registration disabled. Set REGISTER_H2O_MODEL=true in workshop/.env.")

## Expected Result

The validated customer bundle is registered as the configured immutable custom-model version with format, runtime, and golden-validation tags.

Next: `03_create_environment.ipynb`.